# AI온캠 10종 학습 및 평가

팀 Drive는 읽기 전용으로 연결합니다. GPU 사용이 가능한 경우 런타임 유형에서 선택한 뒤 첫 셀부터 실행합니다. 입력 ZIP에는 상품 변형과 준비 코드가 들어 있습니다. 출력 ZIP을 기존 앱 폴더에 풀면 실행용 데이터가 들어갑니다.

In [ ]:
from pathlib import Path
from google.colab import drive, files
import json

drive.mount('/content/drive', readonly=True)
mount = Path('/content/drive')
roots = [mount / 'MyDrive']
if (mount / 'Shareddrives').is_dir():
    roots.extend(p for p in (mount / 'Shareddrives').iterdir() if p.is_dir())
team = next((p for r in roots for p in (r, r / '온캠퍼스_6조', *r.glob('*/온캠퍼스_6조'))
             if p.name == '온캠퍼스_6조' and p.is_dir()), None)
if team is None:
    raise FileNotFoundError('팀 Drive 계정으로 연결한 뒤 이 셀을 다시 실행하세요.')
print('팀 폴더:', team)


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'ultralytics==8.3.228', 'transformers==4.48.0'])


In [ ]:
from google.colab import files
from pathlib import Path
from zipfile import ZipFile
import shutil, sys

inputs = Path('/content/aioncamp_inputs')
inputs.mkdir(exist_ok=True)
if not all((inputs / name).is_file() for name in ('prepare_data.py', 'train_models.py', 'products.zip')):
    if not Path('AI온캠_Colab_입력.zip').is_file():
        files.upload()
    with ZipFile('AI온캠_Colab_입력.zip') as bundle:
        for name in ('prepare_data.py', 'train_models.py', 'products.zip'):
            with bundle.open(name) as source, (inputs / name).open('wb') as target:
                shutil.copyfileobj(source, target)
sys.path.insert(0, str(inputs))


In [ ]:
from train_models import execute

artifact = execute(team, inputs / 'products.zip', Path('/content/aioncamp_final'), epochs=18)
files.download(str(artifact))
